# 🌟 Omni-RecSys Masterclass: Complete End-to-End Walkthrough

Welcome to the **Omni-RecSys Mega Platform** hands-on research notebook.
This notebook covers:
1. Multi-domain dataset inspection (Movies, Music, E-Commerce, Short Feed)
2. Training & evaluating all 6 algorithm generations:
   - **Gen 1**: Time-Decayed Popularity & Item-Item Collaborative Filtering
   - **Gen 2**: Implicit ALS (Hu-Koren 2008) & BPR-MF (Rendle 2009)
   - **Gen 3**: Two-Tower Deep Retrieval (YouTube 2016) with ANN Vector Search
   - **Gen 4**: Wide & Deep, DeepFM, DIN (Alibaba 2018), and MMoE Multi-Task Ranking
   - **Gen 5**: SASRec Sequential Transformer (UCSD 2018) & LightGCN (NUS 2020)
   - **Gen 6**: LinUCB Contextual Bandit (Netflix Artwork Personalization) & LLM Reranker
3. Executing the 5-Stage Production Pipeline with dynamic Netflix page generation
4. Running the Model Arena Head-to-Head Tournament

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from backend.data.loaders import dataset_manager
from backend.models import get_model, MODEL_REGISTRY
from backend.pipeline.orchestrator import ProductionPipeline
from backend.evaluation.arena import ModelArena
from backend.bandits.linucb import LinUCBBandit

print("✅ Omni-RecSys core libraries loaded successfully!")
print(f"Available algorithms ({len(MODEL_REGISTRY)}):", list(MODEL_REGISTRY.keys()))

## 1. Load Multi-Domain Dataset (Movies Domain)

In [ ]:
dataset = dataset_manager.load_domain('movies')
print(f"Items catalog size: {len(dataset.items_df)}")
print(f"Users pool size:    {len(dataset.users_df)}")
print(f"Total interactions: {len(dataset.interactions_df)}")

dataset.items_df[['item_id', 'title', 'primary_genre', 'rating', 'director']].head(5)

## 2. Train SASRec Sequential Transformer (Gen 5)

In [ ]:
sasrec_model = get_model('sasrec', domain='movies')
sasrec_model.fit(dataset.interactions_df, dataset.items_df)

recs = sasrec_model.recommend(user_id='usr_0001', n=5)
for r in recs:
    print(f"#{r.rank} | {r.title:<35} | Genre: {r.category:<12} | Score: {r.score:.3f}")

## 3. Run the 5-Stage Production Pipeline

In [ ]:
from backend.api.recommendations import get_pipeline

pipeline = get_pipeline('movies')
result = pipeline.execute(
    user_id='usr_0001',
    domain='movies',
    catalog_size=len(dataset.items_df),
)

print(f"=== Pipeline Finished in {result.total_latency_ms:.2f} ms ===")
for stage in result.stage_metrics:
    print(f"Stage {stage.stage_number}: {stage.stage_name:<25} | Output: {stage.output_count:3d} items | Latency: {stage.latency_ms:6.2f} ms")

## 4. Run Model Arena Tournament

In [ ]:
from backend.api.recommendations import get_domain_models

models = get_domain_models('movies')
arena = ModelArena(dataset, models)
tournament = arena.run_tournament(model_keys=['popularity', 'implicit_als', 'two_tower', 'sasrec', 'lightgcn'], k=5, sample_users=15)

df_results = pd.DataFrame([m.model_dump() for m in tournament.models])
df_results[['model_name', 'generation', 'ndcg_at_10', 'hit_rate_at_10', 'diversity_ils', 'avg_latency_ms']]